<a href="https://colab.research.google.com/github/nadeemaadi451-droid/nadeemaadi1128-/blob/main/GPT_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-20 09:54:28--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.3’

input.txt.3         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-11-20 09:54:28 (22.8 MB/s) - ‘input.txt.3’ saved [1115394/1115394]



In [45]:
with open('/content/input.txt','r',encoding='utf-8') as f:
  text= f.read()


In [46]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
stoi={ch:i for i, ch in enumerate(chars)}
itos={i:ch for i, ch in enumerate(stoi)}
stoi, itos


({'\n': 0,
  ' ': 1,
  '!': 2,
  '$': 3,
  '&': 4,
  "'": 5,
  ',': 6,
  '-': 7,
  '.': 8,
  '3': 9,
  ':': 10,
  ';': 11,
  '?': 12,
  'A': 13,
  'B': 14,
  'C': 15,
  'D': 16,
  'E': 17,
  'F': 18,
  'G': 19,
  'H': 20,
  'I': 21,
  'J': 22,
  'K': 23,
  'L': 24,
  'M': 25,
  'N': 26,
  'O': 27,
  'P': 28,
  'Q': 29,
  'R': 30,
  'S': 31,
  'T': 32,
  'U': 33,
  'V': 34,
  'W': 35,
  'X': 36,
  'Y': 37,
  'Z': 38,
  'a': 39,
  'b': 40,
  'c': 41,
  'd': 42,
  'e': 43,
  'f': 44,
  'g': 45,
  'h': 46,
  'i': 47,
  'j': 48,
  'k': 49,
  'l': 50,
  'm': 51,
  'n': 52,
  'o': 53,
  'p': 54,
  'q': 55,
  'r': 56,
  's': 57,
  't': 58,
  'u': 59,
  'v': 60,
  'w': 61,
  'x': 62,
  'y': 63,
  'z': 64},
 {0: '\n',
  1: ' ',
  2: '!',
  3: '$',
  4: '&',
  5: "'",
  6: ',',
  7: '-',
  8: '.',
  9: '3',
  10: ':',
  11: ';',
  12: '?',
  13: 'A',
  14: 'B',
  15: 'C',
  16: 'D',
  17: 'E',
  18: 'F',
  19: 'G',
  20: 'H',
  21: 'I',
  22: 'J',
  23: 'K',
  24: 'L',
  25: 'M',
  26: 'N',
  27:

In [47]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib .pyplot as plt

In [48]:
encode= lambda s: [stoi[i] for i in s]
decode= lambda l:''.join( [itos[i] for i in l])
print(encode("Nadeem"))
print(decode(encode("Nadeem")))
data= torch.tensor(encode(text), dtype=torch.long)


[26, 39, 42, 43, 43, 51]
Nadeem


In [49]:
n=int( 0.9 * len(data))
train_data= data[:n]
val_data= data[n:]

In [50]:
context_size= 64
batch_size=16
n_embed= 128
num_head= 4
head_size=16
device= 'cuda' if torch.cuda.is_available() else 'cpu'
dropout= 0.2

In [51]:
def get_batch(split):
  data= train_data if split=='train' else val_data
  ix= torch.randint(len(data)-context_size, (batch_size,)) # randint(low, high, dimension)
  # so the index is 1D tensor with size 4
  x= torch.stack([data[i: i+context_size] for i in ix] )
  y= torch.stack([data[i+1: i+context_size+1] for i in ix])
  # so ix has 4 values and for each value we get 8 values
  return x,y
x, y=get_batch('train')
print(x)
print(y)
x.shape,y.shape


tensor([[56, 43,  1,  ..., 57,  1, 58],
        [39, 58,  1,  ...,  1, 61, 47],
        [52, 41, 43,  ..., 47, 57,  1],
        ...,
        [59, 56, 58,  ..., 54, 43, 56],
        [58, 46, 47,  ..., 53, 58,  1],
        [ 1, 52, 53,  ..., 21, 13, 10]])
tensor([[43,  1, 47,  ...,  1, 58, 53],
        [58,  1, 52,  ..., 61, 47, 58],
        [41, 43,  1,  ..., 57,  1, 51],
        ...,
        [56, 58, 46,  ..., 43, 56, 39],
        [46, 47, 57,  ..., 58,  1, 53],
        [52, 53,  1,  ..., 13, 10,  0]])


(torch.Size([16, 64]), torch.Size([16, 64]))

In [52]:
class SingleHead_Attention(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.head_size= head_size
    self.key=nn.Linear(n_embed, head_size,bias=False)
    self.query=nn.Linear(n_embed, head_size, bias=False)
    self.value=nn.Linear(n_embed, head_size, bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(context_size,context_size)))
    self.dropout= nn.Dropout(dropout)
  def forward(self,x):
    q=self.query(x)
    k= self.key(x)
    v=self.value(x)
    B,T,C=x.shape
    wei= (q @ k.transpose(-2,-1))/(k.ndim**0.5) # B,T,C @ B C T= B T T
    wei=wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
    wei=F.softmax(wei, dim=-1)
    out=wei @ v
    return out

In [53]:
class MultiHead(nn.Module):
  def __init__(self, num_head, head_size):
    super().__init__()
    self.num_head= num_head
    self.head_size= head_size
    self.heads=nn.ModuleList([SingleHead_Attention(head_size) for _ in range(num_head)] )
    self.proj=nn.Linear(num_head * head_size, n_embed)
    self.dropout = nn.Dropout(dropout)
  def forward(self,x):
    out= torch.cat([head(x) for head in self.heads], dim=-1)
    out=self.proj(out)
    out= self.dropout(out)
    return out

In [54]:
class Feed_Forward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.layers= nn.Sequential(
        nn.Linear(n_embed, 4*n_embed, bias=False), nn.ReLU(),
        nn.Linear(4*n_embed, n_embed, bias=False), nn.Dropout(dropout)
    )
  def forward(self,x):
    out= self.layers(x)
    return out


In [55]:
class Transformer(nn.Module):
  def __init__(self, n_embed, head_size):
    super().__init__()
    # MultiHead expects: (num_head, head_size)
    self.multihead=MultiHead(num_head, head_size)
    self.feed_forward= Feed_Forward(n_embed)
    self.norm1=nn.LayerNorm(n_embed)
    self.norm2=nn.LayerNorm(n_embed)
  def forward(self,x):
    x=x+self.multihead(self.norm1(x))
    x=x+self.feed_forward(self.norm2(x))
    return x

In [56]:
class Gpt_LLM(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.layer=2
    self.token_embed=nn.Embedding(vocab_size, n_embed)
    self.position_embed=nn.Embedding(context_size, n_embed)
    self.block= nn.Sequential(*(Transformer(n_embed, head_size) for _ in range(self.layer)))
    self.norm=nn.LayerNorm(n_embed)
    self.lm_head=nn.Linear(n_embed, vocab_size)
  def forward(self,idx, targets=None):
    B,T=idx.shape
    token= self.token_embed(idx) # B,T, n_embed
    pos_enc=self.position_embed(torch.arange(T, device=idx.device)) # T,n_embed
    x= token+ pos_enc #B,T,n_embed
    x=self.block(x)
    x=self.norm(x)
    logits=self.lm_head(x) #B,T,vocab_size
    if targets is None:
      loss=None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      targets=targets.view(B*T)
      loss=F.cross_entropy(logits, targets)
    return logits, loss
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond=idx[:,-context_size:]
      logits, loss=self(idx_cond)
      logits=logits[:,-1,:]
      probs=F.softmax(logits, dim=-1)
      idx_next=torch.multinomial(probs, num_samples=1)
      idx=torch.cat((idx, idx_next), dim=1)
    return idx

In [57]:
# Re-define model with the updated classes and run forward pass
model = Gpt_LLM(vocab_size)
logits, loss = model(x, y)
print("Logits shape:", logits.shape)
print("Loss:", loss.item())

Logits shape: torch.Size([1024, 65])
Loss: 4.322624206542969


In [58]:
import torch

# Configure device and move the model to device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model.to(device)

# Define missing hyperparameters and configurations
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 1e-3

# Define the optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Helper to fetch batches and move them to the active device
def get_batch_on_device(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - context_size, (batch_size,))
    x = torch.stack([data[i:i+context_size] for i in ix])
    y = torch.stack([data[i+1:i+context_size+1] for i in ix])
    return x.to(device), y.to(device)

# Define missing estimate_loss function
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch_on_device(split)
            logits, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Training loop execution
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data using the device-aware helper
    xb, yb = get_batch_on_device('train')

    # evaluate the loss and optimize
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

Using device: cuda
step 0: train loss 4.3144, val loss 4.3219
step 500: train loss 2.1754, val loss 2.2114
step 1000: train loss 1.9435, val loss 2.0290
step 1500: train loss 1.8103, val loss 1.9309
step 2000: train loss 1.7382, val loss 1.8797
step 2500: train loss 1.6935, val loss 1.8486
step 3000: train loss 1.6521, val loss 1.8190
step 3500: train loss 1.6348, val loss 1.7956
step 4000: train loss 1.6058, val loss 1.7720
step 4500: train loss 1.5848, val loss 1.7614
step 4999: train loss 1.5550, val loss 1.7340


VINCENTIO:
I know your reper to son, torters hand our you.
Stizen, or fain's grer; you to is to give fit I say
on the lown, with lyss for that is; who! I think there,
But break hold prove hences of Clark.

Seconce.

MENENIUS:
Bed, gainst cuself: I now make youd, roth
Of with therefores of
Marcius, am heave manner. I'll him morth:
Uport; I will: connell; ithe fiew as thund wits
Should upon of very must see, or knear with mark myss!
reebess thy, tay, I ereing merm'd,
Ere to 